# ChromaDB vs CyborgDB Vec2Text Vulnerability Comparison

> **Last validated: 2026-06-04 | CyborgDB v0.18 (disk-backed)**

This notebook demonstrates the vec2text attack on both ChromaDB and CyborgDB side-by-side.

CyborgDB 0.18 ships as a single-node encrypted index that persists to local disk as append-only segment logs. The attack surface for an exfiltrated CyborgDB store is the on-disk segment files; this notebook reads those files raw, just like an attacker with filesystem access would.
 
### Attack Chain:
1. Sensitive texts → OpenAI embeddings → Store in BOTH ChromaDB and CyborgDB
2. Extract raw bytes from each database backend (ChromaDB SQLite, CyborgDB on-disk segment logs)
3. Use vec2text to attempt reconstruction of original sensitive text
4. Compare results: ChromaDB (vulnerable) vs CyborgDB (protected)

**NOTE**: You will need an OpenAI API key. Set it as your `OPENAI_API_KEY` environment variable or you will be prompted in cell `#1`.

In [1]:
# 0. Install dependencies
# Uninstall sentence-transformers first to avoid conflict with older transformers
%pip uninstall -y sentence-transformers --quiet

# Pinned to the versions validated on 2026-06-10 (Python 3.11, macOS arm64).
# - numpy 2.4.3 ships wheels for Python 3.11-3.14; 1.26.4 has none past 3.12, so
#   Colab's newer Python would compile it from source.
# - transformers 4.57.6 / tokenizers 0.22.2 are vec2text 0.0.13's working pair.
%pip install --quiet \
    "numpy==2.4.3" \
    "transformers==4.57.6" \
    "tokenizers==0.22.2" \
    "vec2text==0.0.13" \
    "openai==2.41.0" \
    "chromadb==1.5.9" \
    "getpass4==0.0.14.1" \
    "cyborgdb-core==0.18.0"

# CPU-only torch 2.11.0 matches the torchvision/torchaudio Colab preinstalls (a newer torch
# breaks their compiled ops), and is >=2.6 for the CVE-2025-32434 fix in torch.load.
%pip install --quiet "torch==2.11.0" --index-url https://download.pytorch.org/whl/cpu

# Colab imports some of these (numpy in particular) at startup. If step 0 replaced one that
# was already loaded, restart so the next cells don't mix old and new versions in memory.
import sys, importlib.metadata
_stale = [n for n in ("numpy", "huggingface_hub", "tokenizers", "transformers", "torch")
          if n in sys.modules and sys.modules[n].__version__ != importlib.metadata.version(n)]
if _stale:
    print(f"🔄 Restarting the runtime to load the updated {', '.join(_stale)}. Then run all cells again.")
    get_ipython().kernel.do_shutdown(restart=True)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Set up OpenAI embedding & vec2text corrector models

import os
import time

print("[1/5] Importing vec2text (pulls in torch/transformers — can take 30-60s on first run)...", flush=True)
_t = time.time()
import vec2text
print(f"      done in {time.time() - _t:.1f}s", flush=True)

print("[2/5] Importing OpenAI client...", flush=True)
from openai import OpenAI
import getpass
print("      done", flush=True)

# Environment variable setup
print("[3/5] Setting thread/MPS env vars...", flush=True)
def setup_env():
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1' 
    os.environ['MKL_NUM_THREADS'] = '1'
    os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
    os.environ['NUMEXPR_NUM_THREADS'] = '1'
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
setup_env()
print("      done", flush=True)

# ANSI color codes for live demo
class Colors:
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    END = '\033[0m'

def print_colored(text, color="", bold=False):
    """Print colored text for demo"""
    prefix = Colors.BOLD if bold else ""
    prefix += getattr(Colors, color.upper(), "")
    print(f"{prefix}{text}{Colors.END}")

# OpenAI setup
print("[4/5] Configuring OpenAI client...", flush=True)
embedding_model = "text-embedding-ada-002"
openai_api_key = (os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter OPENAI_API_KEY: ")).strip()
if not openai_api_key.isascii():
    raise ValueError("OPENAI_API_KEY contains non-ASCII characters (often a smart dash or quote picked up "
                     "while copying). Copy the key directly from the OpenAI dashboard and re-run this cell.")
os.environ["OPENAI_API_KEY"] = openai_api_key
openai_client = OpenAI()
print(f"      OpenAI client ready (embedding model: {embedding_model})", flush=True)

# Load vec2text corrector for inversion
print(f"[5/5] Loading vec2text corrector for OpenAI {embedding_model}...", flush=True)
print("      First run downloads ~1GB of model weights from HuggingFace.", flush=True)
print("      Subsequent runs load from the local HF cache (~/.cache/huggingface).", flush=True)
print("      This may take a few minutes depending on your hardware...", flush=True)
_t = time.time()
corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")
print(f"      Corrector loaded in {time.time() - _t:.1f}s — ready to invert.", flush=True)

[1/5] Importing vec2text (pulls in torch/transformers — can take 30-60s on first run)...


/opt/homebrew/Caskroom/miniconda/base/envs/cyborgdb-core/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


      done in 3.3s
[2/5] Importing OpenAI client...
      done
[3/5] Setting thread/MPS env vars...
      done
[4/5] Configuring OpenAI client...
      OpenAI client ready (embedding model: text-embedding-ada-002)
[5/5] Loading vec2text corrector for OpenAI text-embedding-ada-002...
      First run downloads ~1GB of model weights from HuggingFace.
      Subsequent runs load from the local HF cache (~/.cache/huggingface).
      This may take a few minutes depending on your hardware...
      Corrector loaded in 12.8s — ready to invert.


In [3]:
# 2. Set up ChromaDB and CyborgDB

import os
import shutil
import secrets
import uuid

import chromadb
from cyborgdb_core import Client, StorageConfig


# ChromaDB setup
persist_directory = "./tmp/chromadb_vuln_demo"
chroma_client = chromadb.PersistentClient(path=persist_directory)
try:
    chroma_client.delete_collection("sensitive_docs")
except Exception:
    pass
chroma_collection = chroma_client.create_collection("sensitive_docs")
print(f"✓ ChromaDB ready: {persist_directory}")

# CyborgDB setup — single-node disk-backed encrypted index
CYBORGDB_DISK_PATH = "./tmp/cyborgdb_vuln_demo"
if os.path.exists(CYBORGDB_DISK_PATH):
    shutil.rmtree(CYBORGDB_DISK_PATH)
os.makedirs(CYBORGDB_DISK_PATH, exist_ok=True)

cyborgdb_client = Client(storage_config=StorageConfig.disk(CYBORGDB_DISK_PATH))

# cyborgdb-core requires the KEK on every op (upsert / delete_index / etc.),
# so hold onto it. dimension/metric auto-detect from the first upsert call.
cyborg_index_key = secrets.token_bytes(32)
cyborg_index = cyborgdb_client.create_index(
    index_name=f"comparison-demo-{uuid.uuid4()}",
    index_key=cyborg_index_key,
)
print(f"✓ CyborgDB ready: {CYBORGDB_DISK_PATH} (disk-backed)")


✓ ChromaDB ready: ./tmp/chromadb_vuln_demo
✓ CyborgDB ready: ./tmp/cyborgdb_vuln_demo (disk-backed)


In [4]:
# 3. Define sensitive documents for demo

sensitive_documents = [
    # Financial/Identity Information
    "My social security number is 818-88-8118",
    "Password for admin account is SecurePass123!",
    
    # Medical Information
    "Patient diagnosed with Type 2 diabetes, prescribed metformin",
    "Blood test results show cholesterol level of 277 mg/dL",
    
    # Personal Information
    "John Doe lives at 1 World Trade Center, New York, NY 10007",
    "Phone number +1(212) 653-0688; email info@cyborg.co"
]

In [5]:
# 4. Generate embeddings & store in BOTH databases

print("="*80)
print_colored("GENERATING EMBEDDINGS FROM SENSITIVE DOCUMENTS", bold=True)
print("="*80 + "\n")

def get_embeddings_openai(text_list):
    """Get embeddings from OpenAI API"""
    response = openai_client.embeddings.create(
        input=text_list,
        model=embedding_model,
        encoding_format="float",
    )
    return [e.embedding for e in response.data]

# Get embeddings (generate once, use for both databases)
embeddings = get_embeddings_openai(sensitive_documents)
print(f"Generated {len(embeddings)} embeddings of dimension {len(embeddings[0])}\n")

# Store in ChromaDB
chroma_collection.add(
    documents=sensitive_documents,
    embeddings=embeddings,
    ids=[f"sensitive_doc_{i}" for i in range(len(sensitive_documents))],
    metadatas=[{"type": "sensitive", "doc_num": i} for i in range(len(sensitive_documents))]
)
print("✓ Stored in ChromaDB")

# Store in CyborgDB — cyborgdb-core requires the KEK on every op.
items = [
    {"id": f"sensitive_doc_{i}", "vector": embedding, "contents": doc}
    for i, (doc, embedding) in enumerate(zip(sensitive_documents, embeddings))
]
cyborg_index.upsert(items, index_key=cyborg_index_key)
print("✓ Stored in CyborgDB (encrypted)")

GENERATING EMBEDDINGS FROM SENSITIVE DOCUMENTS

Generated 6 embeddings of dimension 1536

✓ Stored in ChromaDB
✓ Stored in CyborgDB (encrypted)


In [6]:
# 5. Extract embeddings from ChromaDB SQLite backend

import sqlite3
import struct

print("\n" + "="*80)
print_colored("EXTRACTING EMBEDDINGS FROM CHROMADB", bold=True)
print("="*80 + "\n")

# Connect to ChromaDB's SQLite database
db_path = os.path.join(persist_directory, "chroma.sqlite3")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get embedding data from database
cursor.execute("SELECT * FROM embeddings_queue ORDER BY seq_id")
rows = cursor.fetchall()

# Extract embeddings
chroma_extracted = []
for i, row in enumerate(rows):
    doc_id = row[4]
    embedding_blob = row[5]
    num_floats = len(embedding_blob) // 4
    embedding_values = struct.unpack(f'{num_floats}f', embedding_blob)
    chroma_extracted.append(list(embedding_values))
    print_colored(f"Extracted ChromaDB embedding {i+1}: {doc_id}, {len(embedding_values)} dimensions", "RED")

print(f"\n✓ Extracted {len(chroma_extracted)} embeddings from ChromaDB")


EXTRACTING EMBEDDINGS FROM CHROMADB

Extracted ChromaDB embedding 1: sensitive_doc_0, 1536 dimensions
Extracted ChromaDB embedding 2: sensitive_doc_1, 1536 dimensions
Extracted ChromaDB embedding 3: sensitive_doc_2, 1536 dimensions
Extracted ChromaDB embedding 4: sensitive_doc_3, 1536 dimensions
Extracted ChromaDB embedding 5: sensitive_doc_4, 1536 dimensions
Extracted ChromaDB embedding 6: sensitive_doc_5, 1536 dimensions

✓ Extracted 6 embeddings from ChromaDB


In [7]:
# 6. Extract embeddings from CyborgDB on-disk store
#
# An attacker with filesystem access can read CyborgDB's on-disk files
# directly. The strongest version of this demo parses the storage engine's
# record framing and pulls *individual* encrypted values straight from the
# keystore — one ciphertext blob per upserted vector, with no framing.
#
# CyborgDB 0.18 stores each index's backing stores (vectors/contents/id/
# metadata_index) as append-only segment logs (`NNNNNNNN.log`) under
# CYBORGDB_DISK_PATH/idx_<id>/. Each record is a 24-byte header
# (magic "APND", length, crc32c, flags, id) followed by the stored value.
# We find the `vectors` keystore and walk its records.
#
# If the record walk fails, we fall back to an entropy-scan over raw bytes.

import re
import shutil
import struct
import tempfile
from collections import Counter

import numpy as np
import pathlib

print("\n" + "="*80)
print_colored("EXTRACTING EMBEDDINGS FROM CYBORGDB ON-DISK STORE", bold=True)
print("="*80 + "\n")

EMBED_DIM = 1536
EMBED_BYTES = EMBED_DIM * 4  # float32

APPENDIX_MAGIC = 0x41504E44  # "APND"
APPENDIX_HEADER = struct.Struct("<IIIIQ")  # magic, length, crc32c, flags, id
APPENDIX_TOMBSTONE = 0x1


def shannon_entropy(buf: bytes) -> float:
    if not buf:
        return 0.0
    counts = np.bincount(np.frombuffer(buf, dtype=np.uint8), minlength=256)
    probs = counts[counts > 0] / len(buf)
    return float(-np.sum(probs * np.log2(probs)))


def extract_via_segment_logs(disk_path: str, needed: int):
    """Snapshot the on-disk store and walk the `vectors` keystore's segment
    logs record by record. Each value returned is an individual encrypted
    record."""
    attacker_copy = pathlib.Path(tempfile.mkdtemp(prefix="cyborgdb_exfil_"))
    shutil.copytree(disk_path, attacker_copy, dirs_exist_ok=True)
    print(f"  Snapshot copied to: {attacker_copy}")

    segments = sorted(attacker_copy.rglob("vectors/*.log"))
    if not segments:
        raise RuntimeError(f"No vectors/*.log segments under {attacker_copy}")
    print(f"  Found {len(segments)} segment log(s) in the 'vectors' keystore:")
    for seg in segments:
        print(f"    • {seg.relative_to(attacker_copy)}  ({seg.stat().st_size:,} bytes)")

    lengths = []
    out = []
    for seg in segments:
        blob = seg.read_bytes()
        pos = 0
        while pos + APPENDIX_HEADER.size <= len(blob):
            magic, length, _crc, flags, record_id = APPENDIX_HEADER.unpack_from(blob, pos)
            if magic != APPENDIX_MAGIC or length < APPENDIX_HEADER.size or pos + length > len(blob):
                break
            value = blob[pos + APPENDIX_HEADER.size:pos + length]
            pos += length
            if flags & APPENDIX_TOMBSTONE:
                continue
            lengths.append(len(value))
            if len(out) >= needed:
                continue
            # Be generous on length — CEI envelopes ciphertext with a nonce
            # and AEAD tag (we observed 6172 = 6144 + 28 bytes, consistent
            # with AES-GCM's 12B IV + 16B tag).
            if abs(len(value) - EMBED_BYTES) > 64:
                continue
            chunk = value[:EMBED_BYTES] if len(value) >= EMBED_BYTES else \
                    value.ljust(EMBED_BYTES, b"\x00")
            arr = np.frombuffer(chunk, dtype=np.float32).copy()
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            out.append(arr.tolist())
            print_colored(
                f"  Extracted CyborgDB encrypted record {len(out)}: "
                f"id={record_id:016x}, value_len={len(value)} bytes "
                f"[ENCRYPTED — direct segment read]",
                "GREEN",
            )

    if lengths:
        print(f"  Total records in this keystore: {len(lengths)}")
        print(f"  Value-length histogram (top 5): {Counter(lengths).most_common(5)}")
    return out


def extract_via_entropy_scan(disk_path: str, needed: int):
    """Fallback: walk segment log files raw, sample high-entropy windows."""
    DATA_FILE_RE = re.compile(r"^\d+\.log$")
    SCAN_STRIDE = 256
    ENTROPY_FLOOR = 7.0

    def file_score(p: pathlib.Path) -> float:
        try:
            return shannon_entropy(p.read_bytes()[:1 << 20])
        except OSError:
            return 0.0

    all_files = [p for p in pathlib.Path(disk_path).rglob("*")
                 if p.is_file() and p.stat().st_size >= EMBED_BYTES]
    candidate_files = sorted(
        (p for p in all_files if DATA_FILE_RE.search(p.name)),
        key=file_score, reverse=True,
    )
    print(f"  {len(all_files)} file(s) on disk, "
          f"{len(candidate_files)} match N.log pattern")

    out = []
    for path in candidate_files:
        if len(out) >= needed:
            break
        blob = path.read_bytes()
        offset = 0
        while offset <= len(blob) - EMBED_BYTES:
            chunk = blob[offset:offset + EMBED_BYTES]
            if shannon_entropy(chunk) < ENTROPY_FLOOR:
                offset += SCAN_STRIDE
                continue
            arr = np.frombuffer(chunk, dtype=np.float32).copy()
            if not np.all(np.isfinite(arr)):
                offset += SCAN_STRIDE
                continue
            out.append(arr.tolist())
            print_colored(
                f"  Extracted CyborgDB chunk {len(out)}: "
                f"{path.name} @ offset {offset} [ENCRYPTED — entropy scan]",
                "GREEN",
            )
            if len(out) >= needed:
                break
            offset += EMBED_BYTES

    if len(out) < needed:
        remaining = needed - len(out)
        sites = []
        for path in candidate_files:
            size = path.stat().st_size
            step = max(EMBED_BYTES, size // max(needed, 1))
            for offset in range(0, size - EMBED_BYTES + 1, step):
                sites.append((path, offset))
                if len(sites) >= needed * 4:
                    break
            if len(sites) >= needed * 4:
                break
        pick_stride = max(1, len(sites) // remaining) if sites else 1
        for path, offset in sites[::pick_stride][:remaining]:
            blob = path.read_bytes()[offset:offset + EMBED_BYTES]
            arr = np.frombuffer(blob, dtype=np.float32).copy()
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            out.append(arr.tolist())
            print_colored(
                f"  Extracted CyborgDB chunk {len(out)}: "
                f"{path.name} @ offset {offset} [ENCRYPTED — raw window]",
                "GREEN",
            )
    return out


needed = len(sensitive_documents)
cyborg_extracted = []

print_colored("Attempting direct read of the 'vectors' keystore segment logs...", "YELLOW", bold=True)
try:
    cyborg_extracted = extract_via_segment_logs(CYBORGDB_DISK_PATH, needed)
    if cyborg_extracted:
        print_colored(
            f"\n✓ Direct segment read recovered "
            f"{len(cyborg_extracted)} encrypted record(s) — clean ciphertext, no framing.",
            "GREEN", bold=True,
        )
except Exception as e:
    print_colored(
        f"  Direct segment read failed: {type(e).__name__}: {e}",
        "RED",
    )
    print_colored("  Falling back to entropy-scan extraction.\n", "YELLOW")

if len(cyborg_extracted) < needed:
    print_colored(
        f"\nFalling back to entropy-scan for "
        f"{needed - len(cyborg_extracted)} remaining slot(s)...",
        "YELLOW",
    )
    cyborg_extracted.extend(
        extract_via_entropy_scan(CYBORGDB_DISK_PATH, needed - len(cyborg_extracted))
    )

if not cyborg_extracted:
    raise RuntimeError(
        "Could not extract any candidate chunks from disk. "
        "Has cell 4 been run, and does the disk path contain segment logs?"
    )

print(f"\n✓ Recovered {len(cyborg_extracted)} candidate chunk(s) from on-disk store")


EXTRACTING EMBEDDINGS FROM CYBORGDB ON-DISK STORE

Attempting direct read of the 'vectors' keystore segment logs...
  Snapshot copied to: /var/folders/5s/_p9hwlgn43g1srsx1dksx2_h0000gn/T/cyborgdb_exfil_2qk_hqa4
  Found 1 segment log(s) in the 'vectors' keystore:
    • idx_18092459377752120618/vectors/00000001.log  (37,176 bytes)
  Extracted CyborgDB encrypted record 1: id=37e47599f87c41b2, value_len=6172 bytes [ENCRYPTED — direct segment read]
  Extracted CyborgDB encrypted record 2: id=3c5ed41f3b69ed86, value_len=6172 bytes [ENCRYPTED — direct segment read]
  Extracted CyborgDB encrypted record 3: id=c62b506c6d7cca95, value_len=6172 bytes [ENCRYPTED — direct segment read]
  Extracted CyborgDB encrypted record 4: id=fb9de082a2b002ee, value_len=6172 bytes [ENCRYPTED — direct segment read]
  Extracted CyborgDB encrypted record 5: id=1067789385f1c6c3, value_len=6172 bytes [ENCRYPTED — direct segment read]
  Extracted CyborgDB encrypted record 6: id=75271d8c5c929d61, value_len=6172 bytes 

In [8]:
# 7. Define inversion function

import time
import torch

def run_inversion_attack(extracted_embeddings, db_name, color=""):
    """Run vec2text inversion attack on embeddings from a database"""
    
    print("\n" + "="*80)
    print_colored(f"{db_name.upper()}: RUNNING EMBEDDING INVERSION ATTACK", color, bold=True)
    print("="*80)
    
    # Convert to tensor
    embeddings_tensor = torch.tensor(extracted_embeddings, dtype=torch.float32)
    if torch.backends.mps.is_available():
        embeddings_tensor = embeddings_tensor.to('mps')
    elif torch.cuda.is_available():
        embeddings_tensor = embeddings_tensor.cuda()
    
    results = []
    
    for i, (original_doc, embedding) in enumerate(zip(sensitive_documents, embeddings_tensor)):
        print_colored(f"\n[{db_name}] Document #{i+1}:", bold=True)
        print(f"Original:      \"{original_doc}\"")
        
        start_time = time.time()
        reconstructed_list = vec2text.invert_embeddings(
            embeddings=embedding.unsqueeze(0),
            corrector=corrector,
            num_steps=4,
            sequence_beam_width=1,
        )
        reconstructed = reconstructed_list[0]
        inversion_time = time.time() - start_time
        
        print(f"Reconstructed: \"{reconstructed}\"")
        
        # Calculate similarity
        if len(reconstructed) > 0:
            orig_emb_cpu = embedding.cpu()
            new_emb = get_embeddings_openai([reconstructed])[0]
            new_emb_tensor = torch.tensor(new_emb)
            similarity = torch.nn.functional.cosine_similarity(orig_emb_cpu, new_emb_tensor, dim=0).item()
        else:
            similarity = 0
        
        exact_match = original_doc.lower().strip() == reconstructed.lower().strip()
        if exact_match:
            print_colored(f"Exact match!", "RED", bold=True)
        
        sim_color = "RED" if similarity > 0.99 else "YELLOW" if similarity > 0.95 else "GREEN"
        print_colored(f"Similarity: {similarity:.4f}", sim_color, bold=True)
        print(f"Time: {inversion_time:.2f}s")
        
        results.append({
            'original': original_doc,
            'reconstructed': reconstructed,
            'similarity': similarity,
            'time': inversion_time,
            'exact_match': exact_match,
        })
    
    return results

In [9]:
# 8a. Run attack on ChromaDB

# Attack ChromaDB
chroma_results = run_inversion_attack(chroma_extracted, "ChromaDB", "RED")


CHROMADB: RUNNING EMBEDDING INVERSION ATTACK

[ChromaDB] Document #1:
Original:      "My social security number is 818-88-8118"
Reconstructed: "My social security number is 888-808-5811"
Similarity: 0.9659
Time: 3.13s

[ChromaDB] Document #2:
Original:      "Password for admin account is SecurePass123!"
Reconstructed: "Password for admin account is SecurePass123!"
Exact match!
Similarity: 1.0000
Time: 2.51s

[ChromaDB] Document #3:
Original:      "Patient diagnosed with Type 2 diabetes, prescribed metformin"
Reconstructed: "Patient diagnosed with Type 2 diabetes, prescribed metformin"
Exact match!
Similarity: 1.0000
Time: 2.43s

[ChromaDB] Document #4:
Original:      "Blood test results show cholesterol level of 277 mg/dL"
Reconstructed: "Blood test results show cholesterol level of 277 mg/dL"
Exact match!
Similarity: 1.0000
Time: 2.38s

[ChromaDB] Document #5:
Original:      "John Doe lives at 1 World Trade Center, New York, NY 10007"
Reconstructed: "John Doe lives at 1 World Trade C

In [10]:
# 8b. Run attack on CyborgDB

# Attack CyborgDB  
cyborg_results = run_inversion_attack(cyborg_extracted, "CyborgDB", "GREEN")


CYBORGDB: RUNNING EMBEDDING INVERSION ATTACK

[CyborgDB] Document #1:
Original:      "My social security number is 818-88-8118"
Reconstructed: ""
Similarity: 0.0000
Time: 12.98s

[CyborgDB] Document #2:
Original:      "Password for admin account is SecurePass123!"
Reconstructed: ""
Similarity: 0.0000
Time: 10.77s

[CyborgDB] Document #3:
Original:      "Patient diagnosed with Type 2 diabetes, prescribed metformin"
Reconstructed: ""
Similarity: 0.0000
Time: 10.72s

[CyborgDB] Document #4:
Original:      "Blood test results show cholesterol level of 277 mg/dL"
Reconstructed: ""
Similarity: 0.0000
Time: 10.40s

[CyborgDB] Document #5:
Original:      "John Doe lives at 1 World Trade Center, New York, NY 10007"
Reconstructed: ""
Similarity: 0.0000
Time: 10.98s

[CyborgDB] Document #6:
Original:      "Phone number +1(212) 653-0688; email info@cyborg.co"
Reconstructed: ""
Similarity: 0.0000
Time: 10.76s


In [11]:
# 9. Side-by-Side Comparison Summary

print("\n" + "="*80)
print_colored("COMPARISON SUMMARY: ChromaDB vs CyborgDB", bold=True)
print("="*80 + "\n")

def calc_stats(results):
    total = len(results)
    exact = sum(1 for r in results if r['exact_match'])
    high_sim = sum(1 for r in results if r['similarity'] > 0.95)
    avg_sim = np.mean([r['similarity'] for r in results])
    avg_time = np.mean([r['time'] for r in results])
    return {
        'total': total,
        'exact': exact,
        'high_sim': high_sim,
        'avg_sim': avg_sim,
        'avg_time': avg_time
    }

chroma_stats = calc_stats(chroma_results)
cyborg_stats = calc_stats(cyborg_results)

print(f"{'Metric':<35} {'ChromaDB':>15} {'CyborgDB':>15}")
print("="*80)
print(f"{'Total documents':<35} {chroma_stats['total']:>15} {cyborg_stats['total']:>15}")
print(f"{'Exact reconstructions':<35} {chroma_stats['exact']:>15} {cyborg_stats['exact']:>15}")
print(f"{'High similarity (>95%)':<35} {chroma_stats['high_sim']:>15} {cyborg_stats['high_sim']:>15}")
print(f"{'Average similarity':<35} {chroma_stats['avg_sim']*100:>14.2f}% {cyborg_stats['avg_sim']*100:>14.2f}%")
print(f"{'Average inversion time':<35} {chroma_stats['avg_time']:>14.2f}s {cyborg_stats['avg_time']:>14.2f}s")

print("\n" + "="*80)
print_colored("CONCLUSION", bold=True)
print("="*80)

if chroma_stats['avg_sim'] > 0.95:
    print_colored("⚠ ChromaDB: VULNERABLE - High similarity indicates successful reconstruction", "RED", bold=True)
else:
    print_colored("✓ ChromaDB: Protected - Low similarity indicates failed reconstruction", "GREEN", bold=True)

if cyborg_stats['avg_sim'] > 0.95:
    print_colored("⚠ CyborgDB: VULNERABLE - High similarity indicates successful reconstruction", "RED", bold=True)
else:
    print_colored("✓ CyborgDB: PROTECTED - Low similarity indicates failed reconstruction", "GREEN", bold=True)


COMPARISON SUMMARY: ChromaDB vs CyborgDB

Metric                                     ChromaDB        CyborgDB
Total documents                                   6               6
Exact reconstructions                             4               0
High similarity (>95%)                            6               0
Average similarity                           99.34%           0.00%
Average inversion time                        2.77s          11.10s

CONCLUSION
⚠ ChromaDB: VULNERABLE - High similarity indicates successful reconstruction
✓ CyborgDB: PROTECTED - Low similarity indicates failed reconstruction


In [12]:
# 10. Cleanup
chroma_client.delete_collection("sensitive_docs")
cyborg_index.delete_index(index_key=cyborg_index_key)
shutil.rmtree(CYBORGDB_DISK_PATH, ignore_errors=True)